In [2]:
%pip install tf-keras

   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import transformers
import accelerate
import tf_keras

print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("tf-keras:", tf_keras.__version__)

from transformers import Trainer

print("Trainer import: SUCCESS")


Transformers: 4.57.3
Accelerate: 1.10.1
tf-keras: 2.21.0
Trainer import: SUCCESS


In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import torch

print("All libraries imported successfully!")

All libraries imported successfully!


In [6]:
df = pd.read_csv("DS_dataset.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (3150, 4)


,Sentence,Language,Intent,Split
0,"book a table in Montenegro for alissa, corneli...",english,bookrestaurant,train
1,"মন্টিনিগ্রোর অস্টেরিয়াতে অ্যালিসা, কর্নেলিয়া...",bangla,bookrestaurant,train
2,"Montenegro er osteria te alissa, cornelia will...",codemixed,bookrestaurant,train
3,I wanna listen to Electroclash,english,playmusic,train
4,আমি ইলেক্ট্রোক্ল্যাশ শুনতে চাই।,bangla,playmusic,train


In [7]:
print(df.columns)
print(df.info())

Index(['Sentence', 'Language', 'Intent', 'Split'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 3150 entries, 0 to 3149
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Sentence  3150 non-null   str  
 1   Language  3150 non-null   str  
 2   Intent    3150 non-null   str  
 3   Split     3150 non-null   str  
dtypes: str(4)
memory usage: 411.5 KB
None


In [8]:
print(df.isnull().sum())

Sentence    0
Language    0
Intent      0
Split       0
dtype: int64


In [10]:
df = df.dropna(subset=["Sentence", "Intent"])

df["Sentence"] = df["Sentence"].astype(str)
df["Intent"] = df["Intent"].astype(str)

print("Dataset shape:", df.shape)
print(df.isnull().sum())

Dataset shape: (3150, 4)
Sentence    0
Language    0
Intent      0
Split       0
dtype: int64


In [11]:
print("Number of intents:", df["Intent"].nunique())

print("\nIntent distribution:")
print(df["Intent"].value_counts())

Number of intents: 7

Intent distribution:
Intent
getweather              504
playmusic               477
searchcreativework      471
addtoplaylist           444
bookrestaurant          432
ratebook                432
searchscreeningevent    390
Name: count, dtype: int64


In [12]:
print(df["Language"].value_counts())

Language
english      1050
bangla       1050
codemixed    1050
Name: count, dtype: int64


In [13]:
print(df["Split"].value_counts())

Split
train         1908
validation     642
test           600
Name: count, dtype: int64


In [14]:
train_df = df[df["Split"].str.lower() == "train"].copy()
test_df = df[df["Split"].str.lower() == "test"].copy()

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

Training samples: 1908
Testing samples: 600


In [15]:
labels = sorted(df["Intent"].unique())

label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

print("Number of classes:", len(labels))
print("\nLabel mapping:")
print(label2id)

Number of classes: 7

Label mapping:
{'addtoplaylist': 0, 'bookrestaurant': 1, 'getweather': 2, 'playmusic': 3, 'ratebook': 4, 'searchcreativework': 5, 'searchscreeningevent': 6}


In [16]:
train_df["label"] = train_df["Intent"].map(label2id)
test_df["label"] = test_df["Intent"].map(label2id)

print(train_df[["Sentence", "Intent", "label"]].head())

                                            Sentence          Intent  label
0  book a table in Montenegro for alissa, corneli...  bookrestaurant      1
1  মন্টিনিগ্রোর অস্টেরিয়াতে অ্যালিসা, কর্নেলিয়া...  bookrestaurant      1
2  Montenegro er osteria te alissa, cornelia will...  bookrestaurant      1
3                     I wanna listen to Electroclash       playmusic      3
4                    আমি ইলেক্ট্রোক্ল্যাশ শুনতে চাই।       playmusic      3


In [17]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["Sentence", "label"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["Sentence", "label"]]
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['Sentence', 'label', '__index_level_0__'],
    num_rows: 1908
})
Dataset({
    features: ['Sentence', 'label', '__index_level_0__'],
    num_rows: 600
})


In [18]:
from transformers import AutoTokenizer

model_name = "distilbert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

c:\Users\rayha\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rayha\.cache\huggingface\hub\models--distilbert-base-multilingual-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer loaded successfully!


In [19]:
def tokenize_function(examples):
    return tokenizer(
        examples["Sentence"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenization completed!")

Map: 100%|██████████| 600/600 [00:00<00:00, 16629.11 examples/s]

Tokenization completed!


In [20]:
print(train_dataset[0])

{'Sentence': 'book a table in Montenegro for alissa, cornelia williams and katrina at osteria', 'label': 1, '__index_level_0__': 0, 'input_ids': [101, 12748, 169, 21783, 10106, 29251, 10142, 12134, 11253, 117, 93599, 44552, 11337, 26418, 10107, 10111, 27689, 21442, 10160, 10427, 47675, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [21]:
from transformers import AutoModelForSequenceClassification

num_labels = len(labels)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

print("DistilBERT model loaded successfully!")
print("Number of classes:", num_labels)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBERT model loaded successfully!
Number of classes: 7


In [22]:
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(pred):
    predictions = np.argmax(pred.predictions, axis=1)
    labels = pred.label_ids

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distilbert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

print("Training arguments configured!")

Training arguments configured!


In [24]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


In [25]:
trainer.train()

c:\Users\rayha\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.526585,0.906667,0.907027,0.906667,0.906121
2,No log,0.185410,0.958333,0.959394,0.958333,0.958203
3,No log,0.125278,0.973333,0.973283,0.973333,0.973245


c:\Users\rayha\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\rayha\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


TrainOutput(global_step=360, training_loss=0.5707386016845704, metrics={'train_runtime': 956.498, 'train_samples_per_second': 5.984, 'train_steps_per_second': 0.376, 'total_flos': 189577750219776.0, 'train_loss': 0.5707386016845704, 'epoch': 3.0})

In [26]:
results = trainer.evaluate()

print("\nEvaluation Results:")
for key, value in results.items():
    print(f"{key}: {value}")

c:\Users\rayha\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



Evaluation Results:
eval_loss: 0.1252775639295578
eval_accuracy: 0.9733333333333334
eval_precision: 0.9732827105065477
eval_recall: 0.9733333333333334
eval_f1: 0.9732447836484799
eval_runtime: 19.424
eval_samples_per_second: 30.89
eval_steps_per_second: 1.956
epoch: 3.0


In [27]:
distilbert_results = trainer.evaluate(
    eval_dataset=test_dataset
)

print("=" * 60)
print("FINAL DISTILBERT TEST PERFORMANCE")
print("=" * 60)

print(f"Accuracy          : {distilbert_results['eval_accuracy']:.4f}")
print(f"Weighted Precision : {distilbert_results['eval_precision']:.4f}")
print(f"Weighted Recall    : {distilbert_results['eval_recall']:.4f}")
print(f"Weighted F1        : {distilbert_results['eval_f1']:.4f}")

c:\Users\rayha\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


FINAL DISTILBERT TEST PERFORMANCE
Accuracy          : 0.9733
Weighted Precision : 0.9733
Weighted Recall    : 0.9733
Weighted F1        : 0.9732
